# PetroQuest fase 4 — geração QA

Notebook para executar a geração de pergunta e resposta por tópico, registrar métricas de diversidade e comparar regras de similaridade. A rodada padrão busca até 100 questões aceitas por tópico. Saturação e entropia são acompanhadas como diagnóstico; não encerram a geração.

Antes de gerar, baixe os exemplos externos e configure o acesso Azure no servidor.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'pipeline' / 'fase_4' / 'run_qa.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'pipeline'))
from fase_4.qa_prompt import SYSTEM_PROMPT, build_prompt
from fase_3.topicos import TOPICOS
print(f'{len(TOPICOS)} tópicos configurados')

## Rever o prompt antes de iniciar
A célula mostra o texto final enviado ao modelo para uma combinação de tópico, subtópico, dificuldade e exemplos. As questões externas entram como pergunta e resposta, sem alternativas ou passagens.

In [ ]:
topic = next(iter(TOPICOS))
subtopic = next(iter(TOPICOS[topic]))
print(SYSTEM_PROMPT)
print(build_prompt(topic=topic, subtopic=subtopic, difficulty='fácil',
                   format_examples=[{'question': 'Como se determina a densidade de um fluido?', 'answer': 'Massa dividida pelo volume.'}],
                   cross_topic_examples=[{'question': 'Como se verifica a disponibilidade de um sistema?', 'answer': 'Acompanhando o tempo disponível em relação ao período observado.'}],
                   used_concepts=[], count=2))

## Conferir entradas e parâmetros
A execução requer `pipeline/qa_references/data/qa_examples.jsonl`, seeds aprovadas e credenciais Azure válidas. O backend usa esforço `high` para `gpt-5-4-petrobras`.

In [ ]:
refs_path = ROOT / 'pipeline/qa_references/data/qa_examples.jsonl'
seeds_path = ROOT / 'pipeline/qa_references/review/sementes_qa_candidatas.jsonl'
print({'exemplos_externos': refs_path.exists(), 'seeds_especialista': seeds_path.exists()})

## Executar geração
A regra ativa de similaridade cosine começa em 0,50. Para análise, cada questão também recebe decisões simuladas nos cortes 0,70 e 0,85, similaridade lexical Jaccard, trigramas de caracteres, duplicata exata, diversidade lexical e foco conceitual. Entropia é registrada por tópico e não causa parada.

Use `--observe-only` em uma segunda rodada se quiser estimar o volume que cada corte recusaria sem aplicar rejeição por cosine. A rodada principal não deve compartilhar a pasta de saída com essa comparação.

In [ ]:
# Retire o comentário para iniciar a geração no servidor.
# !python pipeline/fase_4/run_qa.py --deployment gpt-5-4-petrobras --effort high --limit-per-topic 100 --similarity-threshold 0.50 --resume

## Analisar resultados
`saida_qa/questoes_qa.jsonl` contém aceitas; `candidatas_metricas.jsonl` guarda aceitas e rejeitadas com métricas e decisões counterfactual; `resumo_topicos.jsonl` registra volume, dificuldade/entropia, métricas agregadas e motivo técnico de término.

A diferença entre os cortes de similaridade pode ser comparada sem nova chamada ao modelo:

In [ ]:
import pandas as pd
metrics_path = ROOT / 'pipeline/fase_4/saida_qa/candidatas_metricas.jsonl'
if metrics_path.exists():
    df = pd.read_json(metrics_path, lines=True)
    decisions = pd.json_normalize(df['counterfactual_decisions'])
    print(pd.DataFrame({
        'candidatas': [len(df)],
        'aceitas_policy_ativa': [int(df['accepted'].sum())],
        'rejeitadas_cosine_050': [int(decisions['cosine_050_reject'].sum())],
        'rejeitadas_cosine_070': [int(decisions['cosine_070_reject'].sum())],
        'rejeitadas_cosine_085': [int(decisions['cosine_085_reject'].sum())],
        'rejeitadas_lexical_050': [int(decisions['lexical_jaccard_050_reject'].sum())],
    }).T)